# Obtaining financial data

This notebook explains how to obtain high-quality price data at daily frequency from US listed stocks using `yfinance`.

**Prerequisites:** Install yfinance if you haven't already:
```bash
pip install yfinance
```

In [28]:
import datetime
import yfinance as yf
import pandas as pd
from pathlib import Path

## Getting data: yfinance and tickers for the S&P 500 universe

This notebook uses `yfinance` to download free historical stock data from Yahoo Finance. While this data is sufficient for pattern recognition experiments, note that:
- It may have minor adjustment issues compared to premium data sources
- It doesn't fully handle survivorship bias for delisted stocks
- There may be occasional data gaps
- Some tickers may fail to download (delisted, symbol changes, etc.)

You will find a list of tickers for the S&P 500 components in the text file `data/SP500_tickers_one_per_line.txt`. Tickers and index components can change over time, so you may want to update this list periodically from sources like:
- https://www.slickcharts.com/sp500
- https://en.wikipedia.org/wiki/List_of_S%26P_500_companies

**Note:** Some ticker symbols may need special handling in yfinance:
- `BRK.B` should be used as-is (yfinance handles it)
- Delisted stocks will show warnings but won't break the download process

In [29]:
# This cell previously showed WRDS interface images - removed as we're using yfinance

## Downloading Single Stock Data with yfinance

The following code demonstrates how to download data for a single stock (e.g., AAPL) using yfinance and save it in a format compatible with the rest of the analysis pipeline.

In [30]:
# Define parameters
start_date = '1980-01-01'  # yfinance reliable from ~1970s for many stocks
end_date = datetime.datetime.now().strftime('%Y-%m-%d')  # or specify end date like '2023-12-31'

# Example: Download data for a single stock
ticker = 'AAPL'  # Change as needed

# Download data with auto_adjust=True to get split-adjusted prices
data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)

# Reset index to make Date a column
data = data.reset_index()

# Rename columns to match CRSP/WRDS format expected by load_data function
data.rename(columns={
    'Date': 'DlyCalDt',
    'Open': 'DlyOpen',
    'High': 'DlyHigh',
    'Low': 'DlyLow',
    'Close': 'DlyClose',
    'Volume': 'DlyVol'
}, inplace=True)

# Convert date to timezone-naive datetime (fixes timezone issues)
if pd.api.types.is_datetime64_any_dtype(data['DlyCalDt']):
    data['DlyCalDt'] = pd.to_datetime(data['DlyCalDt']).dt.tz_localize(None)
else:
    data['DlyCalDt'] = pd.to_datetime(data['DlyCalDt'])

# Add ticker column
data['Ticker'] = ticker

# Calculate price (using Close as the daily price)
data['DlyPrc'] = data['DlyClose']

# Calculate price volume (Close * Volume)
data['DlyPrcVol'] = data['DlyClose'] * data['DlyVol']

# Select and reorder columns to match expected format
data = data[['Ticker', 'DlyCalDt', 'DlyPrc', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyPrcVol']]

# Save to CSV (compressed)
data_dir = Path('./../data')
data_dir.mkdir(exist_ok=True)
output_file = data_dir / f'{ticker}_daily.csv.gz'
data.to_csv(output_file, index=False, compression='gzip')

print(f"Data shape: {data.shape}")
print(f"Date range: {data['DlyCalDt'].min()} to {data['DlyCalDt'].max()}")
print(f"Saved to: {output_file}")
print(data.head())


Data shape: (11386, 9)
Date range: 1980-12-12 00:00:00 to 2026-02-17 00:00:00
Saved to: ../data/AAPL_daily.csv.gz
Price  Ticker   DlyCalDt    DlyPrc   DlyOpen   DlyHigh    DlyLow  DlyClose  \
Ticker                                  AAPL      AAPL      AAPL      AAPL   
0        AAPL 1980-12-12  0.098298  0.098298  0.098725  0.098298  0.098298   
1        AAPL 1980-12-15  0.093169  0.093597  0.093597  0.093169  0.093169   
2        AAPL 1980-12-16  0.086331  0.086758  0.086758  0.086331  0.086331   
3        AAPL 1980-12-17  0.088468  0.088468  0.088895  0.088468  0.088468   
4        AAPL 1980-12-18  0.091033  0.091033  0.091460  0.091033  0.091033   

Price      DlyVol     DlyPrcVol  
Ticker       AAPL                
0       469033600  4.610484e+07  
1       175884800  1.638706e+07  
2       105728000  9.127583e+06  
3        86441600  7.647279e+06  
4        73449600  6.686301e+06  


## Downloading S&P 500 Data with yfinance (TEST MODE: 5 tickers only)

**Note:** This section is set to download only 5 tickers for testing. Change `TEST_MODE = False` and remove the ticker limit to download all S&P 500 tickers.

In [31]:
# TEST MODE: Set to False to download all S&P 500 tickers
TEST_MODE = True
TEST_TICKER_COUNT = 5

# Load S&P 500 tickers from file
ticker_file = Path('./../data/SP500_tickers_one_per_line.txt')
tickers = []

if ticker_file.exists():
    with open(ticker_file, 'r') as f:
        tickers = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(tickers)} tickers from file")
    
    # Limit to 5 tickers for testing
    if TEST_MODE:
        tickers = tickers[:TEST_TICKER_COUNT]
        print(f"TEST MODE: Limited to {len(tickers)} tickers for testing: {tickers}")
else:
    # Fallback: Use test tickers
    tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']
    print(f"Ticker file not found. Using test tickers: {tickers}")

# Define date range (narrowed to 2016-2026 for faster testing)
start_date = '2016-01-01'
end_date = '2026-12-31'  # or use datetime.datetime.now().strftime('%Y-%m-%d') for current date

print(f"Downloading data for {len(tickers)} tickers from {start_date} to {end_date}")
print("This may take a few moments...")


Loaded 503 tickers from file
TEST MODE: Limited to 5 tickers for testing: ['MSFT', 'AAPL', 'AMZN', 'NVDA', 'GOOGL']
This may take a few moments...


In [ ]:
# Download data for all tickers
# Note: For small batches, we download individually for better reliability
all_data_list = []
failed_tickers = []  # Track failed downloads
successful_tickers = []  # Track successful downloads

total_tickers = len(tickers)
print(f"\n{'='*60}")
print(f"Starting download of {total_tickers} tickers...")
print(f"{'='*60}\n")

# For small batches (like 5 tickers), download individually for better reliability
# For larger batches, use batch download
USE_INDIVIDUAL_DOWNLOAD = len(tickers) <= 10

if USE_INDIVIDUAL_DOWNLOAD:
    # Download tickers individually
    print("Using individual download mode (better for small batches)...\n")
    for idx, ticker in enumerate(tickers, 1):
        try:
            print(f"[{idx}/{total_tickers}] Downloading {ticker}...", end=' ', flush=True)
            
            # Download individual ticker
            # Note: auto_adjust=True adjusts prices for splits/dividends
            # Try with auto_adjust first, if that fails, try without
            ticker_data_raw = yf.download(
                ticker, 
                start=start_date, 
                end=end_date, 
                auto_adjust=True, 
                progress=False
            )
            
            # If empty, try without auto_adjust
            if ticker_data_raw.empty:
                print(f"  Trying without auto_adjust...", end=' ')
                ticker_data_raw = yf.download(
                    ticker, 
                    start=start_date, 
                    end=end_date, 
                    auto_adjust=False, 
                    progress=False
                )
            
            if ticker_data_raw.empty:
                print("❌ Failed (empty data)")
                failed_tickers.append(ticker)
                continue
            
            # Handle MultiIndex columns - yfinance returns tuples like ('Close', 'MSFT')
            ticker_data = ticker_data_raw.copy()
            
            # Flatten MultiIndex columns if they exist
            if isinstance(ticker_data.columns, pd.MultiIndex):
                # Extract the first level (the actual column name like 'Close', 'Open', etc.)
                ticker_data.columns = ticker_data.columns.get_level_values(0)
            
            # Reset index to make Date a column
            if isinstance(ticker_data.index, pd.DatetimeIndex):
                ticker_data = ticker_data.reset_index()
                # The index becomes a column, usually named 'Date'
                if ticker_data.columns[0] == 'Date':
                    ticker_data.rename(columns={'Date': 'DlyCalDt'}, inplace=True)
                else:
                    ticker_data.rename(columns={ticker_data.columns[0]: 'DlyCalDt'}, inplace=True)
            else:
                ticker_data = ticker_data.reset_index()
                if 'Date' in ticker_data.columns:
                    ticker_data.rename(columns={'Date': 'DlyCalDt'}, inplace=True)
            
            # Rename OHLCV columns - yfinance returns: Open, High, Low, Close, Volume
            # Handle both regular and MultiIndex flattened columns
            column_map = {}
            for col in ticker_data.columns:
                if col == 'DlyCalDt':
                    continue  # Skip date column
                col_str = str(col).lower() if not isinstance(col, tuple) else str(col[0]).lower()
                if col_str == 'open':
                    column_map[col] = 'DlyOpen'
                elif col_str == 'high':
                    column_map[col] = 'DlyHigh'
                elif col_str == 'low':
                    column_map[col] = 'DlyLow'
                elif col_str in ['close', 'adj close']:
                    column_map[col] = 'DlyClose'
                elif col_str == 'volume':
                    column_map[col] = 'DlyVol'
            
            if not column_map:
                print(f"❌ Failed (could not map columns: {ticker_data.columns.tolist()})")
                failed_tickers.append(ticker)
                continue
            
            # Apply the renaming
            ticker_data.rename(columns=column_map, inplace=True)
            
            # Check that we have the required columns after renaming
            required_cols = ['DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol']
            missing_cols = [col for col in required_cols if col not in ticker_data.columns]
            if missing_cols:
                print(f"❌ Failed (missing columns: {missing_cols}, available: {ticker_data.columns.tolist()})")
                failed_tickers.append(ticker)
                continue
            
            # Remove rows where all OHLCV data is missing (but keep rows with partial data)
            # Only drop rows where Close is missing (most critical)
            ticker_data = ticker_data.dropna(subset=['DlyClose'])
            
            if len(ticker_data) > 0:
                # Convert date to timezone-naive datetime
                if 'DlyCalDt' in ticker_data.columns:
                    if pd.api.types.is_datetime64_any_dtype(ticker_data['DlyCalDt']):
                        ticker_data['DlyCalDt'] = pd.to_datetime(ticker_data['DlyCalDt']).dt.tz_localize(None)
                    else:
                        ticker_data['DlyCalDt'] = pd.to_datetime(ticker_data['DlyCalDt'])
                
                # Add ticker and calculated columns
                ticker_data['Ticker'] = ticker
                ticker_data['DlyPrc'] = ticker_data['DlyClose']
                ticker_data['DlyPrcVol'] = ticker_data['DlyClose'] * ticker_data['DlyVol']
                
                # Reorder columns
                ticker_data = ticker_data[['Ticker', 'DlyCalDt', 'DlyPrc', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyPrcVol']]
                
                all_data_list.append(ticker_data)
                successful_tickers.append(ticker)
                print(f"✅ Success ({len(ticker_data)} records)")
            else:
                print("❌ Failed (no data after cleaning)")
                failed_tickers.append(ticker)
                
        except Exception as e:
            print(f"❌ Failed: {str(e)[:80]}")
            failed_tickers.append(ticker)
            continue

else:
    # Download in batches for larger sets
    batch_size = 50
    for i in range(0, len(tickers), batch_size):
        batch_tickers = tickers[i:i+batch_size]
        batch_num = i//batch_size + 1
        total_batches = (len(tickers)-1)//batch_size + 1
        print(f"[Batch {batch_num}/{total_batches}] Downloading {len(batch_tickers)} tickers...")
        
        try:
            # Download with auto_adjust for split-adjusted prices
            batch_data = yf.download(
                batch_tickers, 
                start=start_date, 
                end=end_date, 
                auto_adjust=True, 
                progress=False,
                group_by='ticker'
            )
            
            # Process each ticker in the batch
            for idx, ticker in enumerate(batch_tickers, 1):
                try:
                    ticker_data = None
                    print(f"  [{idx}/{len(batch_tickers)}] Processing {ticker}...", end=' ')
                    
                    # Check if we have MultiIndex columns (multiple tickers)
                    if isinstance(batch_data.columns, pd.MultiIndex):
                        # Multiple tickers case: columns are (Open, AAPL), (High, AAPL), etc.
                        if ticker in batch_data.columns.levels[1]:
                            ticker_data = pd.DataFrame({
                                'DlyCalDt': batch_data.index,
                                'DlyOpen': batch_data[('Open', ticker)],
                                'DlyHigh': batch_data[('High', ticker)],
                                'DlyLow': batch_data[('Low', ticker)],
                                'DlyClose': batch_data[('Close', ticker)],
                                'DlyVol': batch_data[('Volume', ticker)]
                            })
                        else:
                            failed_tickers.append(ticker)
                            print("❌ Failed (not found in data)")
                            continue
                    else:
                        # Single ticker case
                        ticker_data = batch_data.copy()
                        ticker_data = ticker_data.reset_index()
                        if 'Date' in ticker_data.columns:
                            ticker_data.rename(columns={'Date': 'DlyCalDt'}, inplace=True)
                        
                        column_map = {
                            'Open': 'DlyOpen',
                            'High': 'DlyHigh', 
                            'Low': 'DlyLow',
                            'Close': 'DlyClose',
                            'Volume': 'DlyVol'
                        }
                        ticker_data.rename(columns=column_map, inplace=True)
                    
                    if ticker_data is not None:
                        ticker_data = ticker_data.dropna()
                        
                        if len(ticker_data) > 0:
                            # Convert date to timezone-naive datetime
                            if 'DlyCalDt' in ticker_data.columns:
                                if pd.api.types.is_datetime64_any_dtype(ticker_data['DlyCalDt']):
                                    ticker_data['DlyCalDt'] = pd.to_datetime(ticker_data['DlyCalDt']).dt.tz_localize(None)
                                else:
                                    ticker_data['DlyCalDt'] = pd.to_datetime(ticker_data['DlyCalDt'])
                            
                            ticker_data['Ticker'] = ticker
                            ticker_data['DlyPrc'] = ticker_data['DlyClose']
                            ticker_data['DlyPrcVol'] = ticker_data['DlyClose'] * ticker_data['DlyVol']
                            ticker_data = ticker_data[['Ticker', 'DlyCalDt', 'DlyPrc', 'DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol', 'DlyPrcVol']]
                            
                            all_data_list.append(ticker_data)
                            successful_tickers.append(ticker)
                            print(f"✅ Success ({len(ticker_data)} records)")
                        else:
                            failed_tickers.append(ticker)
                            print("❌ Failed (no data)")
                except Exception as e:
                    print(f"❌ Failed: {str(e)[:50]}")
                    failed_tickers.append(ticker)
                    continue
                    
        except Exception as e:
            print(f"  ❌ Error downloading batch: {e}")
            failed_tickers.extend(batch_tickers)
            continue

# Combine all data
if all_data_list:
    # Flatten any MultiIndex columns before concatenating
    for i, df in enumerate(all_data_list):
        if isinstance(df.columns, pd.MultiIndex):
            all_data_list[i] = df.copy()
            all_data_list[i].columns = df.columns.get_level_values(-1)  # Use last level of MultiIndex
    
    combined_data = pd.concat(all_data_list, ignore_index=True)
    
    # Ensure no MultiIndex in columns
    if isinstance(combined_data.columns, pd.MultiIndex):
        combined_data.columns = combined_data.columns.get_level_values(-1)
    
    # Sort by ticker and date
    combined_data = combined_data.sort_values(['Ticker', 'DlyCalDt']).reset_index(drop=True)
    
    # Save to compressed CSV
    data_dir = Path('./../data')
    data_dir.mkdir(exist_ok=True)
    output_file = data_dir / 'SP500_daily_data_1980_to_2023.csv.gz'
    combined_data.to_csv(output_file, index=False, compression='gzip')
    
    print(f"\n{'='*60}")
    print(f"Download complete!")
    print(f"{'='*60}")
    print(f"Total records: {len(combined_data):,}")
    print(f"Successfully downloaded: {len(successful_tickers)}/{total_tickers} tickers")
    if successful_tickers:
        print(f"  ✅ Successful: {', '.join(successful_tickers)}")
    if failed_tickers:
        print(f"  ❌ Failed ({len(failed_tickers)}): {', '.join(failed_tickers)}")
    print(f"Date range: {combined_data['DlyCalDt'].min()} to {combined_data['DlyCalDt'].max()}")
    print(f"Saved to: {output_file}")
    print(f"\nFirst few rows:")
    print(combined_data.head())
else:
    print("\n❌ No data was downloaded. Please check your ticker list and date range.")



Starting download of 5 tickers...

[Batch 1/1] Downloading 5 tickers...
  [1/5] Processing MSFT... ❌ Failed (not found in data)
  [2/5] Processing AAPL... ❌ Failed (not found in data)
  [3/5] Processing AMZN... ❌ Failed (not found in data)
  [4/5] Processing NVDA... ❌ Failed (not found in data)
  [5/5] Processing GOOGL... ❌ Failed (not found in data)

❌ No data was downloaded. Please check your ticker list and date range.


## Notes on Failed Downloads

Some tickers may fail to download for various reasons:
- **Delisted stocks**: Companies that are no longer publicly traded
- **Symbol changes**: Tickers that have changed (e.g., Facebook → META)
- **Data availability**: Some stocks may not have historical data for the requested date range
- **Special symbols**: Some tickers with special characters (like `BRK.B`) may need special handling

The download process will continue even if some tickers fail. Failed tickers are logged and reported at the end of the download process. You can manually try downloading individual tickers or update your ticker list to exclude problematic symbols.

**To download all S&P 500 tickers:** Set `TEST_MODE = False` in the cell above and remove the ticker limit.

# END